# 3.1 整体架构 (Overall Architecture)

设计高效的神经网络架构，在模型容量、训练稳定性和推理效率之间取得最优平衡。

本节涵盖：
- Decoder-Only Transformer
- Encoder-Only Transformer
- Encoder-Decoder
- 混合架构

## 1. Decoder-Only Transformer

**目的**：生成式语言模型的主流架构

**基本原理**：仅使用Transformer的解码器部分，通过因果注意力掩码（causal mask）实现自回归生成。每个token只能关注自身及之前的token，确保生成过程从左到右依次进行。

**核心特点**：
- 因果掩码：下三角矩阵确保信息只从左向右流动
- 自回归生成：每次预测下一个token，逐步生成完整序列
- 统一架构：预训练和推理使用完全相同的架构

**代表模型**：GPT-4、LLaMA、Mistral、Qwen、DeepSeek

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len=256):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.register_buffer('mask', torch.tril(torch.ones(max_seq_len, max_seq_len)).unsqueeze(0).unsqueeze(0))

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = attn.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(out)

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2, max_seq_len=256):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.layers = nn.ModuleList([DecoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.tok_embed(x) + self.pos_embed(pos)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        return self.lm_head(h)

model = DecoderOnlyTransformer(vocab_size=1000, d_model=128, n_heads=4, n_layers=2)
x = torch.randint(0, 1000, (2, 32))
logits = model(x)
params = sum(p.numel() for p in model.parameters())

print('=== Decoder-Only Transformer ===')
print(f'Input shape: {x.shape}')
print(f'Output logits shape: {logits.shape}')
print(f'Total parameters: {params:,}')

attn_layer = model.layers[0].attn
sample_q = torch.randn(1, 4, 8, 32)
sample_k = torch.randn(1, 4, 8, 32)
raw_scores = (sample_q @ sample_k.transpose(-2, -1)) / math.sqrt(32)
mask = torch.tril(torch.ones(8, 8))
masked_scores = raw_scores.masked_fill(mask == 0, float('-inf'))
print(f'\nCausal mask visualization (4x4):')
print(mask[:4, :4].int().tolist())
print(f'\nEach token can only attend to itself and previous tokens.')
print(f'This ensures autoregressive generation from left to right.')

## 2. Encoder-Only Transformer

**目的**：文本理解与表征学习

**基本原理**：使用双向注意力（bidirectional attention）捕获完整上下文信息。每个token可以关注序列中的所有其他token，包括前方和后方的token。

**核心特点**：
- 双向注意力：无掩码限制，每个token看到完整上下文
- 表征学习：输出每个位置的上下文化表征
- 下游适配：通常接分类头或序列标注头

**代表模型**：BERT、RoBERTa、DeBERTa

**典型任务**：文本分类、命名实体识别、语义检索、句子相似度

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class BidirectionalAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.d_head)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, D)
        return self.proj(out)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = BidirectionalAttention(d_model, n_heads)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class EncoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2, n_classes=3):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(256, d_model)
        self.layers = nn.ModuleList([EncoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.cls_head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.embed(x) + self.pos_embed(pos)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        cls_repr = h[:, 0]
        return self.cls_head(cls_repr), h

model = EncoderOnlyTransformer(vocab_size=1000, d_model=128, n_heads=4, n_layers=2, n_classes=3)
x = torch.randint(0, 1000, (4, 32))
cls_logits, hidden_states = model(x)

print('=== Encoder-Only Transformer ===')
print(f'Input shape: {x.shape}')
print(f'CLS logits shape: {cls_logits.shape} (for classification)')
print(f'Hidden states shape: {hidden_states.shape} (contextual representations)')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

attn_layer = model.layers[0].attn
sample = torch.randn(1, 4, 128)
with torch.no_grad():
    qkv = attn_layer.qkv(sample).reshape(1, 4, 3, 4, 32).permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]
    attn_weights = torch.softmax((q @ k.transpose(-2, -1)) / math.sqrt(32), dim=-1)
print(f'\nBidirectional attention pattern (head 0, token 0):')
print(f'  Attends to ALL positions: {attn_weights[0, 0, 0].softmax(-1).tolist()[:4]}...')
print(f'\nKey difference: No causal mask -> every token sees the full context.')

## 3. Encoder-Decoder

**目的**：序列到序列任务

**基本原理**：编码器处理输入序列（双向注意力），解码器自回归生成输出序列（因果注意力），编码器-解码器之间通过交叉注意力连接。

**核心特点**：
- 编码器：双向注意力，完整理解输入
- 解码器：因果掩码 + 交叉注意力，自回归生成
- 交叉注意力：解码器的每个位置可以关注编码器的所有输出

**代表模型**：T5、BART、mBART

**典型任务**：机器翻译、文本摘要、代码生成

In [ ]:
import torch
import torch.nn as nn
import math

torch.manual_seed(42)

class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x, enc_output):
        B, T_dec, D = x.shape
        T_enc = enc_output.size(1)
        q = self.q_proj(x).reshape(B, T_dec, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(enc_output).reshape(B, T_enc, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(enc_output).reshape(B, T_enc, self.n_heads, self.d_head).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        attn = torch.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T_dec, D)
        return self.proj(out)

class EncoderDecoderTransformer(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4, n_layers=2, max_seq_len=256):
        super().__init__()
        self.enc_embed = nn.Embedding(vocab_size, d_model)
        self.dec_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)

        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)

        self.cross_attn = CrossAttention(d_model, n_heads)
        self.self_attn = CausalSelfAttention(d_model, n_heads, max_seq_len)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Linear(d_model * 4, d_model))
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, src, tgt):
        T_src, T_tgt = src.size(1), tgt.size(1)
        src_pos = torch.arange(T_src, device=src.device)
        tgt_pos = torch.arange(T_tgt, device=tgt.device)

        enc_h = self.encoder(self.enc_embed(src) + self.pos_embed(src_pos))

        tgt_h = self.dec_embed(tgt) + self.pos_embed(tgt_pos)
        tgt_h = tgt_h + self.self_attn(self.norm1(tgt_h))
        tgt_h = tgt_h + self.cross_attn(self.norm2(tgt_h), enc_h)
        tgt_h = tgt_h + self.ffn(self.norm3(tgt_h))

        return self.lm_head(tgt_h)

model = EncoderDecoderTransformer()
src = torch.randint(0, 1000, (2, 20))
tgt = torch.randint(0, 1000, (2, 15))
logits = model(src, tgt)

print('=== Encoder-Decoder Transformer ===')
print(f'Source input: {src.shape} (encoder processes this with bidirectional attention)')
print(f'Target input: {tgt.shape} (decoder generates with causal + cross attention)')
print(f'Output logits: {logits.shape} (next token predictions)')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

print(f'\nInformation flow:')
print(f'  1. Encoder: src -> bidirectional attention -> enc_output')
print(f'  2. Decoder self-attn: tgt -> causal attention -> tgt_context')
print(f'  3. Decoder cross-attn: tgt_context x enc_output -> combined')
print(f'  4. FFN + LM head -> next token prediction')

## 4. 混合架构

**目的**：兼顾不同任务需求

**基本原理**：将上述架构进行组合或变体设计，以同时支持理解和生成任务，或通过混合专家等方式提升模型容量。

**典型混合架构**：
- **UL2**：统一预训练范式，结合前缀语言建模和去噪自编码，同一模型支持多种预训练目标
- **Switch Transformer**：将MoE与Encoder-Decoder架构结合，通过稀疏激活大幅增加参数量
- **Jamba**：结合Transformer层和Mamba（SSM）层，兼顾注意力机制的长程建模和SSM的线性复杂度

**设计原则**：
- 根据任务需求选择合适的注意力模式
- 通过MoE/SSM等组件提升效率
- 统一架构减少部署复杂度

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

class HybridArchitectureDemo(nn.Module):
    def __init__(self, vocab_size=1000, d_model=128, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(256, d_model)

        self.attn_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.ssm_layer = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.SiLU(),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x, use_attn=True):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.embed(x) + self.pos_embed(pos)

        if use_attn:
            h = h + self.attn_layer(h)
        else:
            h = h + self.ssm_layer(self.norm1(h))

        h = self.norm2(h)
        return self.lm_head(h)

model = HybridArchitectureDemo()
x = torch.randint(0, 1000, (2, 32))

logits_attn = model(x, use_attn=True)
logits_ssm = model(x, use_attn=False)

attn_params = sum(p.numel() for p in model.attn_layer.parameters())
ssm_params = sum(p.numel() for p in model.ssm_layer.parameters())

print('=== Hybrid Architecture Demo ===')
print(f'Input: {x.shape}')
print(f'\nAttention branch output: {logits_attn.shape}')
print(f'SSM branch output: {logits_ssm.shape}')
print(f'\nAttention layer params: {attn_params:,}')
print(f'SSM layer params: {ssm_params:,}')
print(f'Complexity: Attention=O(n^2), SSM=O(n)')

print(f'\n=== Real-World Hybrid Architectures ===')
print(f'UL2: Unified pretraining combining prefix-LM + denoising')
print(f'Switch Transformer: MoE + encoder-decoder for massive scaling')
print(f'Jamba: Alternating Transformer + Mamba layers')
print(f'  -> Attention for precise recall, SSM for efficient long-context')